# Notebook 1 — Data exploration and splitting

**Computer Vision CW1 — Diabetic Retinopathy Stage Detection**
Mohamed Shafran · NIBM (BSCCOMP24.2P)

This notebook is the first step of the project. It does four things:

1. **Finds the APTOS 2019 data** and loads the labels.
2. **Measures every image** — size, brightness, contrast and sharpness — so the dataset section of the report is based on numbers, not guesses.
3. **Finds duplicate and near-duplicate images**, which the dataset is known to contain.
4. **Writes one fixed train / validation / test split** to `splits.csv`. Every later notebook reads that file, so all experiments are compared on exactly the same data.

**How to run it:** set *Accelerator* to **None** (no GPU needed here — it saves your weekly quota), attach the APTOS 2019 competition data, then **Run All**. It takes a few minutes, most of it reading the 3,662 images once.

**No model is trained here, and the test split is never looked at again until the final evaluation.**

In [ ]:
# ============================================================
# Setup: imports, fixed seed, output folders, one shared style
# ============================================================
import os, io, json, time, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

SEED = 42                      # fixed, so the split is identical every time this runs
np.random.seed(SEED)

OUT_DIR = Path("/kaggle/working")          # Kaggle keeps what we save here
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]

# One colour scheme for every figure in the project.
# The five stages are ORDERED (0 -> 4 = healthy -> worst), so they get a single
# hue from light to dark rather than five unrelated colours. Splits are not
# ordered, so they get three distinct hues.
STAGE_COLORS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]
SPLIT_COLORS = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}
INK, INK_SOFT, MUTED, GRID, FLAG = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#d03b3b"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK_SOFT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "font.size": 10, "figure.dpi": 110, "savefig.bbox": "tight",
})

def tidy(ax):
    """Remove chart junk: drop the top/right frame, put a hairline grid behind the data."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    return ax

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| pandas", pd.__version__)

## 1. Find the data and load the labels

Kaggle mounts competition data under a folder whose name we should not assume — on this account it sits at `/kaggle/input/competitions/aptos2019-blindness-detection/`. Instead of hard-coding that path, the function below searches for the folder that actually contains `train.csv`. If Kaggle changes the layout, the notebook still works.

`train.csv` has one row per image: `id_code` (the file name without `.png`) and `diagnosis` (the stage, 0–4).

In [ ]:
def find_data_dir(root="/kaggle/input"):
    """Return the folder that holds train.csv, wherever Kaggle mounted it."""
    for path, dirs, files in os.walk(root):
        if "train.csv" in files:
            return Path(path)
        # never crawl into the image folders while searching - thousands of files
        dirs[:] = [d for d in dirs if not d.lower().endswith("_images")]
    raise FileNotFoundError("train.csv not found under /kaggle/input")

DATA = find_data_dir()
TRAIN_IMG_DIR = DATA / "train_images"
print("Data folder:", DATA)

labels = pd.read_csv(DATA / "train.csv")
labels["path"] = labels["id_code"].map(lambda code: str(TRAIN_IMG_DIR / f"{code}.png"))

missing = [p for p in labels["path"] if not os.path.exists(p)]
print("rows in train.csv :", len(labels))
print("image files found :", len(labels) - len(missing))
assert not missing, f"{len(missing)} images listed in train.csv are missing from disk"

labels.head()

## 2. Class distribution

The first thing to establish is how unbalanced the dataset is, because it shapes almost every later decision: which metric we judge the model by, whether we weight the classes, and how much confidence the rare-stage results deserve.

In [ ]:
counts = labels["diagnosis"].value_counts().sort_index()
dist = pd.DataFrame({
    "stage":   counts.index,
    "name":    [CLASS_NAMES[i] for i in counts.index],
    "images":  counts.values,
    "percent": (100 * counts.values / len(labels)).round(1),
})
print(dist.to_string(index=False))
print("\ntotal images       :", len(labels))
print(f"largest / smallest : {counts.max()/counts.min():.1f}x "
      f"({CLASS_NAMES[int(counts.idxmax())]} vs {CLASS_NAMES[int(counts.idxmin())]})")

In [ ]:
# Figure 1 - how many images each stage has
fig, ax = plt.subplots(figsize=(7.2, 4.2))
bars = ax.bar(dist["stage"], dist["images"], width=0.66, color=STAGE_COLORS)
for bar, n, pct in zip(bars, dist["images"], dist["percent"]):
    ax.text(bar.get_x() + bar.get_width()/2, n + counts.max()*0.02,
            f"{n}\n{pct}%", ha="center", va="bottom", fontsize=9, color=INK_SOFT)

ax.set_xticks(dist["stage"])
ax.set_xticklabels([f"{s}\n{n}" for s, n in zip(dist["stage"], dist["name"])])
ax.set_ylabel("Images")
ax.set_ylim(0, counts.max() * 1.20)
ax.set_title("APTOS 2019 - images per diabetic retinopathy stage")
tidy(ax)
plt.savefig(FIG_DIR / "fig01_class_distribution.png")
plt.show()

## 3. Measure every image

Each image is read **once** and five things are recorded:

| Measurement | What it tells us |
|---|---|
| width, height | how much the cameras differ, and how much resizing we will do |
| brightness | mean grey value — very low means an underexposed photo |
| contrast | spread of grey values — low means a flat, washed-out photo |
| sharpness | variance of the Laplacian: a standard blur measure. Sharp edges give a high value, a blurred photo a low one |
| md5 and dHash | fingerprints used to find duplicates in the next section |

Two speed details worth knowing:

- `cv2.IMREAD_REDUCED_COLOR_4` decodes the image at a quarter of its size. That is far faster and is plenty for statistics and fingerprints. The true width and height still come from the file header, through PIL.
- Every image is then resized to the same 256×256 before measuring, so the numbers are comparable across cameras of different resolution.

**dHash (difference hash) — additional knowledge.** Shrink the grey image to 9×8 pixels and compare each pixel with its right-hand neighbour: brighter gives a 1, darker a 0. That is 8 rows × 8 comparisons = 64 bits, packed into 8 bytes. Two copies of the same photo produce the same fingerprint; a re-saved or slightly altered copy differs in only a few bits. It is about ten lines of our own code rather than an external library.

In [ ]:
def dhash(gray):
    """64-bit difference hash of a greyscale image (see the explanation above)."""
    small = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA)
    return np.packbits(small[:, 1:] > small[:, :-1])      # 8 x 8 comparisons -> 8 bytes

def scan_image(path):
    """Read one image once and return everything we need from it."""
    raw = open(path, "rb").read()

    with Image.open(io.BytesIO(raw)) as im:               # header only, no full decode
        width, height = im.size

    small = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_REDUCED_COLOR_4)
    if small is None:
        raise ValueError(f"could not decode {path}")

    standard = cv2.resize(small, (256, 256), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(standard, cv2.COLOR_BGR2GRAY)

    return {
        "width": width, "height": height,
        "megapixels": round(width * height / 1e6, 2),
        "file_mb": round(len(raw) / 1e6, 3),
        "brightness": float(gray.mean()),                            # 0 = black, 255 = white
        "contrast": float(gray.std()),
        "sharpness": float(cv2.Laplacian(gray, cv2.CV_64F).var()),   # low = blurred
        "md5": hashlib.md5(raw).hexdigest(),                         # identical FILES
        "dhash": dhash(gray).tobytes().hex(),                        # identical PICTURES
    }

In [ ]:
# Read all images. Threads help here because OpenCV releases the interpreter lock
# while decoding, so several images are decoded at the same time.
start = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    records = list(pool.map(scan_image, labels["path"]))

stats = pd.concat([labels.reset_index(drop=True), pd.DataFrame(records)], axis=1)
print(f"scanned {len(stats)} images in {time.time() - start:.0f} s")

stats[["width", "height", "megapixels", "file_mb",
       "brightness", "contrast", "sharpness"]].describe().round(1)

In [ ]:
# Figure 2 - what sizes do the images come in?
sizes = (stats.groupby(["width", "height"]).size()
         .reset_index(name="count").sort_values("count", ascending=False))
print("distinct image sizes:", len(sizes))
print(sizes.head(5).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.scatter(sizes["width"], sizes["height"],
           s=20 + 240 * sizes["count"] / sizes["count"].max(),
           color="#2a78d6", alpha=0.55, edgecolor="#fcfcfb", linewidth=0.8)
for _, row in sizes.head(3).iterrows():
    ax.annotate(f'{int(row.width)}x{int(row.height)}  ({int(row["count"])})',
                (row.width, row.height), textcoords="offset points",
                xytext=(9, 6), fontsize=9, color=INK_SOFT)

ax.set_xlabel("Width (pixels)")
ax.set_ylabel("Height (pixels)")
ax.set_title("Image sizes in the dataset (bubble area = how many images)")
tidy(ax)
ax.grid(axis="x", color=GRID, linewidth=0.8)
plt.savefig(FIG_DIR / "fig02_image_sizes.png")
plt.show()

In [ ]:
# Figure 3 - five random images from each stage
fig, axes = plt.subplots(5, 5, figsize=(12.5, 13))
for row in range(5):
    picks = stats[stats["diagnosis"] == row].sample(5, random_state=SEED)
    for col, (_, r) in enumerate(picks.iterrows()):
        image = cv2.cvtColor(cv2.imread(r["path"], cv2.IMREAD_REDUCED_COLOR_8), cv2.COLOR_BGR2RGB)
        ax = axes[row, col]
        ax.imshow(image)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_color(STAGE_COLORS[row]); spine.set_linewidth(2.5)
        if col == 0:
            ax.set_ylabel(f"{row} - {CLASS_NAMES[row]}", fontsize=10, color=INK)

fig.suptitle("Five random images per stage", fontsize=13, fontweight="bold", color=INK)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig03_samples_per_class.png")
plt.show()

### Image quality

Real screening photos vary: some are out of focus, some underexposed. The competition organisers say so themselves. Rather than repeat that claim, we measure it and keep the numbers for the report's dataset limitations section. Nothing is deleted here — the flags are used later, in the error analysis, to check whether the model's mistakes concentrate on poor images.

In [ ]:
# Flag the worst 5% by sharpness and by brightness
BLUR_CUT   = stats["sharpness"].quantile(0.05)
DARK_CUT   = stats["brightness"].quantile(0.05)
low_quality = (stats["sharpness"] <= BLUR_CUT) | (stats["brightness"] <= DARK_CUT)

print(f"blurriest 5% cut-off : sharpness <= {BLUR_CUT:.0f}")
print(f"darkest 5% cut-off   : brightness <= {DARK_CUT:.0f}")
print(f"images flagged       : {int(low_quality.sum())} ({100*low_quality.mean():.1f}%)")

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.scatter(stats.loc[~low_quality, "brightness"], stats.loc[~low_quality, "sharpness"],
           s=14, color="#2a78d6", alpha=0.35, linewidth=0, label="Normal")
ax.scatter(stats.loc[low_quality, "brightness"], stats.loc[low_quality, "sharpness"],
           s=16, color=FLAG, alpha=0.8, linewidth=0, label="Dark or blurred (worst 5%)")
ax.set_yscale("log")
ax.set_xlabel("Brightness (mean grey value)")
ax.set_ylabel("Sharpness (log scale)")
ax.set_title("Image quality - most images are usable, with a dark and blurred tail")
ax.legend(frameon=False, labelcolor=INK_SOFT)
tidy(ax)
plt.savefig(FIG_DIR / "fig04_image_quality.png")
plt.show()

In [ ]:
# Figure 5 - the six least sharp images, as evidence of what "low quality" means here
worst = stats.nsmallest(6, "sharpness")
fig, axes = plt.subplots(1, 6, figsize=(15, 3.2))
for ax, (_, r) in zip(axes, worst.iterrows()):
    ax.imshow(cv2.cvtColor(cv2.imread(r["path"], cv2.IMREAD_REDUCED_COLOR_8), cv2.COLOR_BGR2RGB))
    ax.set_title(f'{CLASS_NAMES[int(r["diagnosis"])]}\nsharpness {r["sharpness"]:.0f}',
                 fontsize=9, color=INK_SOFT)
    ax.axis("off")
fig.suptitle("The six least sharp images in the dataset", fontsize=12, fontweight="bold", color=INK)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig05_worst_quality.png")
plt.show()

## 4. Duplicates

APTOS is known to contain repeated images. This matters for one reason: if a photo appears in training **and** in the test set, the model can score well by recognising a picture it has already seen, and our reported results would be too optimistic. That is data leakage.

Two checks run here:

1. **Byte-identical files** — the md5 of the file contents. Same bytes, same image.
2. **Near-identical pictures** — dHash fingerprints that differ in at most 5 bits out of 64. This catches copies that were re-saved, slightly cropped or re-compressed.

The Hamming distance (how many bits differ) between **every pair** of images is computed in one matrix multiplication rather than with a loop over ~6.7 million pairs.

Images that are linked — directly, or through a third image — are then collected into **groups** with a small union-find structure. Whole groups move together in the split, so a picture can never appear on both sides.

In [ ]:
# --- 1. byte-identical files -------------------------------------------------
duplicate_files = stats["md5"].duplicated(keep=False)
print("images that are byte-identical to another image:", int(duplicate_files.sum()))

# --- 2. near-identical pictures ---------------------------------------------
hashes = np.array([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in stats["dhash"]])
bits = np.unpackbits(hashes, axis=1).astype(np.int16)     # (N, 64) matrix of 0/1

# Hamming distance for every pair at once:
#   places where A=1 and B=0   +   places where A=0 and B=1
distance = bits @ (1 - bits).T + (1 - bits) @ bits.T
np.fill_diagonal(distance, 999)                            # ignore each image against itself

HAMMING_CUT = 5                                            # out of 64 bits
pair_list = [tuple(map(int, p)) for p in np.argwhere(np.triu(distance <= HAMMING_CUT, k=1))]
print(f"near-duplicate pairs (<= {HAMMING_CUT} bits different):", len(pair_list))

In [ ]:
# --- 3. group the copies together -------------------------------------------
# Union-find: every image starts in its own group; each matching pair merges two
# groups, so a chain of copies (A~B, B~C) ends up as one group {A, B, C}.
parent = list(range(len(stats)))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]        # path compression keeps this fast
        i = parent[i]
    return i

def union(i, j):
    root_i, root_j = find(i), find(j)
    if root_i != root_j:
        parent[root_j] = root_i

for i, j in pair_list:
    union(i, j)

stats["group_id"] = [find(i) for i in range(len(stats))]

group_sizes = stats.groupby("group_id").size()
print("groups of 2 or more images :", int((group_sizes > 1).sum()))
print("images inside such a group :", int(group_sizes[group_sizes > 1].sum()))
print("largest group              :", int(group_sizes.max()))

# --- 4. do the copies agree on the stage? -----------------------------------
stages_per_group = stats.groupby("group_id")["diagnosis"].nunique()
conflict_groups  = stages_per_group[stages_per_group > 1].index
conflict_mask    = stats["group_id"].isin(conflict_groups)
print("groups whose copies disagree about the stage:",
      len(conflict_groups), f"({int(conflict_mask.sum())} images)")

In [ ]:
# Figure 6 - what the duplicates actually look like (disagreeing pairs first)
disagreeing = [(i, j) for i, j in pair_list
               if stats["diagnosis"].iloc[i] != stats["diagnosis"].iloc[j]]
agreeing    = [p for p in pair_list if p not in set(disagreeing)]
to_show     = (disagreeing + agreeing)[:3]

if to_show:
    fig, axes = plt.subplots(len(to_show), 2, figsize=(7, 3.4 * len(to_show)), squeeze=False)
    for r, (i, j) in enumerate(to_show):
        for c, index in enumerate((i, j)):
            row = stats.iloc[index]
            axes[r, c].imshow(cv2.cvtColor(cv2.imread(row["path"], cv2.IMREAD_REDUCED_COLOR_8),
                                           cv2.COLOR_BGR2RGB))
            axes[r, c].set_title(f'{row["id_code"]}\nstage {int(row["diagnosis"])} - '
                                 f'{CLASS_NAMES[int(row["diagnosis"])]}',
                                 fontsize=9, color=INK_SOFT)
            axes[r, c].axis("off")
    fig.suptitle("Duplicate / near-duplicate examples", fontsize=12, fontweight="bold", color=INK)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig06_duplicate_examples.png")
    plt.show()
else:
    print("no near-duplicate pairs at this threshold - nothing to show")

## 5. The split

The dataset is divided once: **70% train, 15% validation, 15% test**.

- **Stratified** — each split keeps the same stage proportions as the whole dataset, so the rare stages are represented everywhere.
- **Group-aware** — the split is made over duplicate *groups*, not over images, so copies always stay on the same side.
- **Fixed** — one seed, saved to `splits.csv`, reused by every later notebook. Experiments are then comparable, and the test set stays untouched until the very end.
- Groups whose copies disagree about the stage are **dropped**, because we cannot tell which label is correct. The number dropped goes in the report.

The `assert` lines below are deliberate: if a later edit ever breaks the grouping, the notebook stops here instead of quietly producing a leaky split.

In [ ]:
from sklearn.model_selection import train_test_split

clean = stats[~conflict_mask].copy()
print(f"images kept: {len(clean)}   (dropped {len(stats) - len(clean)} with disagreeing labels)")

# one row per GROUP - this is what gets split
groups = (clean.groupby("group_id")
          .agg(diagnosis=("diagnosis", "first"), n_images=("id_code", "size"))
          .reset_index())
print("groups to split:", len(groups))

train_groups, rest = train_test_split(
    groups, test_size=0.30, stratify=groups["diagnosis"], random_state=SEED)
val_groups, test_groups = train_test_split(
    rest, test_size=0.50, stratify=rest["diagnosis"], random_state=SEED)

split_of = {}
for frame, name in ((train_groups, "train"), (val_groups, "val"), (test_groups, "test")):
    split_of.update({gid: name for gid in frame["group_id"]})
clean["split"] = clean["group_id"].map(split_of)

# --- checks that must pass ---------------------------------------------------
assert clean["split"].notna().all(), "some images were not assigned to a split"
assert clean.groupby("group_id")["split"].nunique().max() == 1, \
    "a duplicate group ended up in more than one split"
assert set(clean["split"]) == {"train", "val", "test"}

print("\nimages per split:")
print(clean["split"].value_counts().reindex(["train", "val", "test"]).to_string())

In [ ]:
# Stage proportions per split - this is the proof that stratification worked
pivot = (clean.pivot_table(index="diagnosis", columns="split",
                           values="id_code", aggfunc="count")
         .reindex(columns=["train", "val", "test"]).fillna(0).astype(int))
percent = (100 * pivot / pivot.sum()).round(1)

print("images per stage and split:")
print(pivot.to_string())
print("\npercent within each split:")
print(percent.to_string())

x, width = np.arange(5), 0.26
fig, ax = plt.subplots(figsize=(7.6, 4.4))
for k, split in enumerate(["train", "val", "test"]):
    bars = ax.bar(x + (k - 1) * width, percent[split], width=width - 0.03,
                  color=SPLIT_COLORS[split], label=split)
    for bar, value in zip(bars, percent[split]):
        ax.text(bar.get_x() + bar.get_width()/2, value + 0.7, f"{value:.0f}",
                ha="center", fontsize=8, color=INK_SOFT)

ax.set_xticks(x)
ax.set_xticklabels([f"{i}\n{CLASS_NAMES[i]}" for i in range(5)])
ax.set_ylabel("Percent of that split")
ax.set_ylim(0, float(percent.to_numpy().max()) * 1.20)
ax.set_title("Stage proportions match across train, validation and test")
ax.legend(frameon=False, labelcolor=INK_SOFT, ncols=3)
tidy(ax)
plt.savefig(FIG_DIR / "fig07_split_distribution.png")
plt.show()

## 6. Save the results

Three files go to `/kaggle/working`:

- **`splits.csv`** — one row per image: id, stage, split, group, and the measurements. This is the file every later notebook reads.
- **`image_stats_all.csv`** — the full measurements including the dropped images, kept for the report.
- **`dataset_summary.json`** — the headline numbers, so the report quotes the run rather than memory.

Plus the figures in `/kaggle/working/figures/`.

In [ ]:
columns = ["id_code", "diagnosis", "split", "group_id", "width", "height",
           "brightness", "contrast", "sharpness", "md5", "dhash"]
clean[columns].sort_values("id_code").to_csv(OUT_DIR / "splits.csv", index=False)
stats.to_csv(OUT_DIR / "image_stats_all.csv", index=False)

summary = {
    "images_total": int(len(stats)),
    "images_kept": int(len(clean)),
    "class_counts_total": {CLASS_NAMES[int(k)]: int(v) for k, v in counts.items()},
    "byte_identical_images": int(duplicate_files.sum()),
    "near_duplicate_pairs": int(len(pair_list)),
    "duplicate_groups": int((group_sizes > 1).sum()),
    "conflicting_groups_dropped": int(len(conflict_groups)),
    "images_dropped": int(len(stats) - len(clean)),
    "low_quality_flagged": int(low_quality.sum()),
    "split_sizes": {k: int(v) for k, v in clean["split"].value_counts().items()},
    "hamming_threshold": HAMMING_CUT,
    "seed": SEED,
}
with open(OUT_DIR / "dataset_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nfiles written:", sorted(p.name for p in OUT_DIR.glob("*.csv")),
      "+ dataset_summary.json +", len(list(FIG_DIR.glob("*.png"))), "figures")

## 7. What to do next

1. **Save Version → Save & Run All.** This runs the notebook from top to bottom on a clean machine and stores the outputs. Only a saved version's output can be attached to the next notebook.
2. Tell me the printed summary — especially the duplicate counts and the split sizes. Those numbers go straight into the report's dataset section.
3. Download the notebook (**File → Download notebook**) into `CW1_DR_Stage_Detection/kaggle_notebooks/` and commit it.

**For the report**, this notebook produces: the class distribution figure and table, the image size figure, the sample grid, the quality figure and the worst-quality examples, the duplicate examples, and the split proportions figure — with the summary numbers to describe them.

Notebook 2 will take `splits.csv` from here and build the preprocessing pipeline.